In [1]:
import os
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn.functional as F
from sklearn.cluster import KMeans
import zipfile
import csv
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
# Специфичные импорты для Google Colab
try:
    from google.colab import drive
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    from IPython.display import FileLink, display

2026-04-26 15:58:07.584933: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777219087.828260      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777219087.893445      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777219088.468657      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777219088.468694      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777219088.468697      23 computation_placer.cc:177] computation placer alr

In [2]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns # Для красивых палитр

# 1. Трекинг одного двигателя (Кривая деградации во времени)
def plot_engine_prediction(y_true, y_pred, engine_idx, epoch, fd):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(y_true, label='Actual RUL (Ground Truth)', color='blue', linestyle='--', linewidth=2)
    ax.plot(y_pred, label=f'Predicted RUL (Epoch {epoch})', color='red', linewidth=2)
    ax.set_title(f'FD00{fd} - Engine {engine_idx} Degradation Curve', fontsize=12)
    ax.set_xlabel('Cycles (Time)')
    ax.set_ylabel('Remaining Useful Life (RUL)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

# 2. Scatter Plot всех тестовых предсказаний (насколько плотно мы к y=x)
def plot_scatter_rul(y_true, y_pred, epoch, fd):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.4, c='purple', edgecolors='none', s=20)
    ax.plot([0, 125], [0, 125], 'k--', linewidth=2, label='Ideal Prediction (y=x)') # Идеальная диагональ
    ax.set_xlabel('Actual RUL')
    ax.set_ylabel('Predicted RUL')
    ax.set_title(f'FD00{fd} - Scatter Plot (Epoch {epoch})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

# 3. Визуализация Латентов (t-SNE) для доказательства дистилляции
def plot_latent_space_tsne(s_latents, t_latents, epoch, fd):
    # s_latents: (N, d), t_latents: (N, d)
    latents = np.vstack([s_latents, t_latents])

    # t-SNE сжатие до 2D
    tsne = TSNE(n_components=2, random_state=42, perplexity = min(30, len(latents)//3), max_iter=1000)
    reduced = tsne.fit_transform(latents)

    fig, ax = plt.subplots(figsize=(8, 6))
    # Рисуем Студента
    ax.scatter(reduced[:len(s_latents), 0], reduced[:len(s_latents), 1],
               label='Student (Projected Features)', alpha=0.6, c='blue', s=30)
    # Рисуем Учителя
    ax.scatter(reduced[len(s_latents):, 0], reduced[len(s_latents):, 1],
               label='Teacher (Oracle Features)', alpha=0.6, c='red', s=30, marker='x')

    ax.set_title(f'FD00{fd} - Latent Space Alignment (t-SNE) Epoch {epoch}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

In [3]:
# ---------------- Utilities ----------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def nasa_score(y_true, y_pred):
    s = 0.0
    for yt, yp in zip(y_true, y_pred):
        diff = yp - yt
        if diff < 0:
            s += math.exp(-diff / 13.0) - 1.0
        else:
            s += math.exp(diff / 10.0) - 1.0
    return float(s)

def positional_encoding(max_len, d_model, device):
    pe = torch.zeros(max_len, d_model, device=device)
    position = torch.arange(0, max_len, dtype=torch.float32, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32, device=device) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# ---------------- Data helpers ----------------
def load_cmapss(path):
    df = pd.read_csv(path, sep=r'\s+', header=None)
    cols = ["id", "cycle"] + [f"op{i}" for i in range(1, 4)] + [f"s{i}" for i in range(1, 22)]
    df.columns = cols
    selected_sensors = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]
    sensor_cols = [f"s{i}" for i in selected_sensors]
    keep_cols = ["id", "cycle", "op1", "op2", "op3"] + sensor_cols
    df[sensor_cols] = df[sensor_cols].astype(np.float32)
    return df[keep_cols]

def apply_regime_specific_normalization(df_train, df_test, n_clusters=6):
    op_cols = ["op1", "op2", "op3"]
    sensor_cols = [c for c in df_train.columns if c.startswith('s')]
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(df_train[op_cols])

    # Сохраняем кластеры режима (эксплуатационные условия)
    df_train['regime_idx'] = kmeans.predict(df_train[op_cols])
    df_test['regime_idx'] = kmeans.predict(df_test[op_cols])

    df_train = df_train.copy()
    df_test = df_test.copy()

    for cluster_id in range(n_clusters):
        tr_mask = df_train['regime_idx'] == cluster_id
        te_mask = df_test['regime_idx'] == cluster_id
        if tr_mask.sum() == 0: continue

        mins = df_train.loc[tr_mask, sensor_cols].min()
        maxs = df_train.loc[tr_mask, sensor_cols].max()
        diffs = maxs - mins
        diffs[diffs == 0] = 1.0

        df_train.loc[tr_mask, sensor_cols] = (df_train.loc[tr_mask, sensor_cols] - mins) / diffs
        if te_mask.sum() > 0:
            df_test.loc[te_mask, sensor_cols] = (df_test.loc[te_mask, sensor_cols] - mins) / diffs

    return df_train, df_test

def create_train_windows_v4(df, window, future_len=20, max_rul=125, stride=1, fd_num=1):
    X_windows, X_future, y_meta = [], [], []
    sensor_cols = [c for c in df.columns if c.startswith('s')]

    fault_mapping = {}
    if fd_num >= 3:
        # 1. Создаем временные очищенные данные для кластеризации
        # Вычитаем среднее по каждому режиму для каждого датчика
        # Это "схлопывает" влияние операционных условий (6 режимов) в одну точку
        df_temp = df.copy()
        df_temp[sensor_cols] = df.groupby('regime_idx')[sensor_cols].transform(lambda x: x - x.mean())
        
        # 2. Берем "хвосты" очищенных данных (последние 20 циклов)
        # Теперь здесь только чистая сигнатура поломки без примеси режима
        last_states = df_temp.groupby('id').tail(20).groupby('id')[sensor_cols].mean()
        
        # 3. Кластеризуем физику поломки (HPC vs Fan)
        km_fault = KMeans(n_clusters=2, random_state=42, n_init=10).fit(last_states)
        fault_mapping = dict(zip(last_states.index, km_fault.labels_))
        
        del df_temp # Очищаем память

    # Основной цикл нарезки окон
    for eid in df['id'].unique():
        sub = df[df['id']==eid].sort_values('cycle')
        fm = fault_mapping.get(eid, 0) if fd_num >= 3 else 0
        T = len(sub)
        rul_all = np.minimum(np.array([(T-1)-i for i in range(T)]), max_rul)
        sensors = sub[sensor_cols].values
        regimes = sub['regime_idx'].values

        for end in range(window, T+1, stride):
            # Текущее окно (для Студента)
            X_windows.append(sensors[end-window:end, :])
            
            # БУДУЩЕЕ окно (для Учителя)
            # Берем отрезок от конца текущего окна и вперед на future_len
            f_start = end
            f_end = end + future_len
            if f_end <= T:
                fut = sensors[f_start:f_end, :]
            else:
                # Если двигатель развалится раньше, чем через 20 циклов, 
                # дополняем будущее окно последним состоянием (padding)
                pad_len = f_end - T
                last_val = sensors[-1:, :]
                fut = np.vstack([sensors[f_start:T, :], np.repeat(last_val, pad_len, axis=0)])
            
            X_future.append(fut)
            y_meta.append([rul_all[end-1], regimes[end-1], fm])

    return np.stack(X_windows), np.stack(X_future), np.array(y_meta, dtype=np.float32)

def create_test_windows(df, df_rul, window, max_rul):
    r_test = df_rul.values.flatten()
    X_test_list, y_test_list = [], []
    for i, eid in enumerate(df['id'].unique()):
        sub = df[df['id']==eid].sort_values('cycle')
        T = len(sub)
        sensors = sub[[c for c in sub.columns if c.startswith('s')]].values
        if T >= window:
            x = sensors[-window:, :]
        else:
            pad = np.repeat(sensors[0:1,:], window-T, axis=0)
            x = np.vstack([pad, sensors])
        X_test_list.append(x)
        y_test_list.append(min(r_test[i], max_rul))
    return np.stack(X_test_list), np.array(y_test_list, dtype=np.float32)

# НОВАЯ ФУНКЦИЯ: Для правильной отрисовки деградации во времени
def get_engine_trajectory(student, df_test, engine_rul, engine_idx, window, max_rul, device):
    student.eval()
    sub = df_test[df_test['id'] == engine_idx].sort_values('cycle')
    T = len(sub)
    sensors = sub[[c for c in sub.columns if c.startswith('s')]].values

    X_list = []
    for end in range(1, T + 1):
        if end >= window:
            x = sensors[end-window:end, :]
        else:
            pad = np.repeat(sensors[0:1,:], window-end, axis=0)
            x = np.vstack([pad, sensors[:end, :]])
        X_list.append(x)

    X_tensor = torch.tensor(np.stack(X_list), dtype=torch.float32).to(device)

    with torch.no_grad():
        preds = student(X_tensor)
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = preds.clamp(0.0, float(max_rul)).cpu().numpy().flatten()

    final_rul = engine_rul[engine_idx]
    y_true_time = np.array([final_rul + (T - t) for t in range(1, T + 1)])
    y_true_time_clamped = np.minimum(y_true_time, max_rul)

    return y_true_time_clamped, preds

class CMapssWindowDataset(Dataset):
    def __init__(self, X, X_fut, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.X_fut = torch.tensor(X_fut, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.X_fut[idx], self.y[idx]

# ---------------- Modules ----------------
class PatchEmbedDimWise(nn.Module):
    def __init__(self, window, n_sensors, patch_size, d_model, pos_learnable=True):
        super().__init__()
        self.window = window
        self.P = patch_size
        self.n_patches = math.ceil(window / patch_size)
        self.d_model = d_model
        self.patch_proj = nn.Linear(self.P, d_model, bias=True)
        if pos_learnable:
            self.pos_embed = nn.Parameter(torch.zeros(n_sensors, self.n_patches, d_model))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        else:
            self.pos_embed = None

    def forward(self, x):
        B, W, S = x.shape
        pad_len = (self.n_patches * self.P) - W
        if pad_len > 0:
            x = torch.cat([x, x[:, -1:, :].repeat(1, pad_len, 1)], dim=1)
        x = x.view(B, self.n_patches, self.P, S).permute(0, 3, 1, 2).contiguous()
        emb_flat = self.patch_proj(x.view(B * S * self.n_patches, self.P))
        emb = emb_flat.view(B, S, self.n_patches, self.d_model)
        if self.pos_embed is not None:
            emb = emb + self.pos_embed.unsqueeze(0)
        return emb

class STARAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim=256, dropout=0.1):
        super().__init__()
        self.temporal_mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.temporal_norm1 = nn.LayerNorm(d_model)
        self.temporal_ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
        self.temporal_norm2 = nn.LayerNorm(d_model)

        self.sensor_mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.sensor_norm1 = nn.LayerNorm(d_model)
        self.sensor_ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
        self.sensor_norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        B, S, T, d = x.shape
        res = x
        x_flat = x.view(B * S, T, d)
        temp_out, _ = self.temporal_mha(x_flat, x_flat, x_flat)
        x = self.temporal_norm1(res + temp_out.view(B, S, T, d))
        x = self.temporal_norm2(x + self.temporal_ffn(x))

        res = x
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T, S, d)
        sensor_out, _ = self.sensor_mha(x_flat, x_flat, x_flat)
        x = self.sensor_norm1(res + sensor_out.view(B, T, S, d).permute(0, 2, 1, 3))
        x = self.sensor_norm2(x + self.sensor_ffn(x))
        return x

class PatchMerging(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
    def forward(self, x):
        B, S, T, d = x.shape
        if T <= 1: return x
        if T % 2 == 1:
            x = x[:, :, :-1, :]
            T = T - 1
        left = x[:, :, 0::2, :]
        right = x[:, :, 1::2, :]
        merged = torch.cat([left, right], dim=-1)
        return self.proj(merged)

class STAREncoder(nn.Module):
    def __init__(self, n_scales, d_model, nhead, ffn_dim, dropout, n_layers_per_scale=4):
        super().__init__()
        self.n_scales = n_scales
        self.layers = nn.ModuleList([
            nn.ModuleList([STARAttentionBlock(d_model, nhead, ffn_dim, dropout) for _ in range(n_layers_per_scale)])
            for _ in range(n_scales)
        ])
        self.patch_merging = nn.ModuleList([PatchMerging(d_model) for _ in range(n_scales - 1)])

    def forward(self, x):
        features = []
        cur = x
        for i in range(self.n_scales):
            for layer in self.layers[i]:
                cur = layer(cur)
            features.append(cur)
            if i < self.n_scales - 1:
                cur = self.patch_merging[i](cur)
        return features

class DecoderBlockTwoStage(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim=256, dropout=0.05):
        super().__init__()
        self.temporal_mha = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)
        self.temporal_norm = nn.LayerNorm(d_model)
        self.sensor_mha = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)
        self.sensor_norm = nn.LayerNorm(d_model)
        self.cross_mha = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)
        self.cross_norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
        self.ffn_norm = nn.LayerNorm(d_model)

    def forward(self, dec, enc_feat, temporal_causal_mask=None):
        B, S, T_dec, d = dec.shape
        _, _, T_enc, _ = enc_feat.shape

        res = dec
        dec_temp_in = dec.reshape(B * S, T_dec, d)
        temp_out, _ = self.temporal_mha(dec_temp_in, dec_temp_in, dec_temp_in, attn_mask=temporal_causal_mask)
        dec = self.temporal_norm(res + temp_out.reshape(B, S, T_dec, d))

        res = dec
        dec_sensor_in = dec.permute(0, 2, 1, 3).reshape(B * T_dec, S, d)
        sensor_out, _ = self.sensor_mha(dec_sensor_in, dec_sensor_in, dec_sensor_in)
        dec = self.sensor_norm(res + sensor_out.reshape(B, T_dec, S, d).permute(0, 2, 1, 3))

        res = dec
        dec_cross_in = dec.reshape(B, S * T_dec, d)
        enc_cross_kv = enc_feat.reshape(B, S * T_enc, d)
        cross_out, _ = self.cross_mha(dec_cross_in, enc_cross_kv, enc_cross_kv)
        dec = self.cross_norm(res + cross_out.reshape(B, S, T_dec, d))

        res = dec
        dec = self.ffn_norm(res + self.ffn(dec))
        return dec

class STARDecoder(nn.Module):
    def __init__(self, n_scales, d_model, nhead, ffn_dim, dropout, n_layers_per_scale=2):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.ModuleList([DecoderBlockTwoStage(d_model, nhead, ffn_dim, dropout) for _ in range(n_layers_per_scale)])
            for _ in range(n_scales)
        ])

    def forward(self, dec_in, enc_kv, blocks_for_scale):
        cur = dec_in
        for blk in blocks_for_scale:
            cur = blk(cur, enc_kv, temporal_causal_mask=None)
        return cur

class PredictionHead(nn.Module):
    def __init__(self, d_model, ffn_dim, n_scales, dropout):
        super().__init__()
        self.scale_mlps = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
            for _ in range(n_scales)
        ])
        self.final_mlp = nn.Sequential(nn.Linear(d_model * n_scales, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, 1))

    def forward(self, dec_outputs):
        pooled = []
        for i, f in enumerate(dec_outputs):
            v = f.mean(dim=(1, 2))
            v = self.scale_mlps[i](v)
            pooled.append(v)
        cat = torch.cat(pooled, dim=-1)
        out = self.final_mlp(cat)
        return out.view(-1)

# ---------------- Oracle Enabled Full Model ----------------
class STARModelFull(nn.Module):
    def __init__(self, window, n_sensors, d_model, nhead, num_scales,
                 ffn_dim=256, patch_size=4, dropout=0.25,
                 encoder_layers_per_scale=4, decoder_layers_per_scale=2,
                 pos_learnable=True, is_teacher=False, target_noise_std=0.05,
                 num_regimes=6, num_faults=2, future_len=20): # <-- Добавили future_len
        super().__init__()
        self.is_teacher = is_teacher
        self.num_scales = num_scales
        
        self.patch_embed = PatchEmbedDimWise(window, n_sensors, patch_size, d_model, pos_learnable)

        if self.is_teacher:
            # Проектор для будущих сенсоров. 
            # Размер: (future_len * n_sensors) -> d_model
            self.future_proj = nn.Linear(future_len * n_sensors, d_model)
            
            self.regime_emb = nn.Embedding(num_regimes, d_model)
            self.fault_emb = nn.Embedding(num_faults, d_model)
            self.teacher_noise_std = target_noise_std

            nn.init.normal_(self.regime_emb.weight, std=0.02)
            nn.init.normal_(self.fault_emb.weight, std=0.02)
            # На старте Учитель не должен мешать
            nn.init.zeros_(self.future_proj.weight)
            nn.init.zeros_(self.future_proj.bias)
        else:
            self.projectors = nn.ModuleList([
                nn.Sequential(
                    nn.Linear(d_model, d_model),
                    nn.LayerNorm(d_model),
                    nn.GELU(),
                    nn.Linear(d_model, d_model)
                ) for _ in range(num_scales)
            ])

        self.encoder = STAREncoder(num_scales, d_model, nhead, ffn_dim, dropout, encoder_layers_per_scale)
        self.decoder = STARDecoder(num_scales, d_model, nhead, ffn_dim, dropout, decoder_layers_per_scale)
        self.pred_head = PredictionHead(d_model, ffn_dim, num_scales, dropout)

    # ИСПРАВЛЕНИЕ 4: Добавили x_future в аргументы
    def forward(self, x, x_future=None, y_meta=None, return_latents=False):
        emb = self.patch_embed(x)  # [B, S, T, d]
        B, S, T, d = emb.shape

        if self.is_teacher and x_future is not None and y_meta is not None:
            # 1. Обработка будущего окна
            # x_future: [B, future_len, S] -> Flatten: [B, future_len * S]
            fut_flat = x_future.view(B, -1)
            
            # Добавляем шум прямо к будущим сенсорам (чтобы Учитель не переобучался)
            if self.training:
                noise = torch.randn_like(fut_flat) * self.teacher_noise_std
                fut_flat = fut_flat + noise

            # Проецируем в d_model: [B, 1, 1, d]
            fut_bias = self.future_proj(fut_flat).unsqueeze(1).unsqueeze(1)

            # 2. Обработка режимов и поломок
            regime_id = y_meta[:, 1].long()
            fault_id = y_meta[:, 2].long()
            
            r_bias = self.regime_emb(regime_id).unsqueeze(1).unsqueeze(1)
            f_bias = self.fault_emb(fault_id).unsqueeze(1).unsqueeze(1)

            # 3. Инъекция знаний: просто складываем будущее с режимами
            emb = emb + fut_bias + r_bias + f_bias

        # Дальше стандартный код энкодера/декодера
        enc_feats = self.encoder(emb)

        # Шаг 4: Декодер и предсказание RUL
        dec_outs = []
        dec_input = None

        for i in range(len(enc_feats)):
            enc_feat = enc_feats[i]
            B_e, S_e, T_e, d_e = enc_feat.shape
            
            if dec_input is None:
                # Начальный вход декодера — позиционные эмбеддинги
                pe = positional_encoding(T_e, d_e, device=x.device)
                dec_input = pe.unsqueeze(0).unsqueeze(0).repeat(B_e, S_e, 1, 1)

            # Прогон через блок декодера
            cur = self.decoder(dec_input, enc_feat, self.decoder.blocks[i])
            dec_outs.append(cur)
            dec_input = cur

        # Финальный выход (прогноз RUL)
        out = self.pred_head(dec_outs)

        # Шаг 5: Возврат латентов для Distillation Loss
        if return_latents:
            if self.is_teacher:
                # Учитель отдает "чистые" признаки
                return out, enc_feats
            else:
                # Студент прогоняет свои признаки через проекторы
                projected_feats = []
                for i, feat in enumerate(enc_feats):
                    projected_feats.append(self.projectors[i](feat))
                return out, projected_feats

        return out
# ---------------- Training / Evaluation ----------------
def train_one_epoch(student, teacher, loader, optimizer, device, scaler, criterion, cfg, epoch, tb_writer, fd):
    student.train()

    # 1. Задаем параметры расписания в зависимости от сложности датасета (fd)
    if fd in [1]:
        # Простые датасеты (быстро учится, нужна тишина в конце)
        peak_fraction = 0.5     # Пик влияния Учителя на 40% обучения
        freeze_fraction = 0.75  # Заморозка на 75%
        min_lambda = 150.0       # Сильно ослабляем Учителя в конце
    if fd in [2]:
        peak_fraction = 0.5     # Пик влияния Учителя на 40% обучения
        freeze_fraction = 0.75  # Заморозка на 75%
        min_lambda = 100.0       # Сильно ослабляем Учителя в конце
    if fd in [3]:
        peak_fraction = 0.5     # Пик влияния Учителя на 40% обучения
        freeze_fraction = 0.75  # Заморозка на 75%
        min_lambda = 75.0       # Сильно ослабляем Учителя в конце
    if fd in [4]:
        peak_fraction = 0.5     # Пик влияния Учителя на 40% обучения
        freeze_fraction = 0.75  # Заморозка на 75%
        min_lambda = 100.0       # Сильно ослабляем Учителя в конце

    # 2. Заморозка / Разморозка Учителя
    warmup_epochs = max(1, int(cfg["epochs"] * freeze_fraction))
    if epoch > warmup_epochs:
        teacher.eval()
        for param in teacher.parameters():
            param.requires_grad = False
    else:
        teacher.train()
        # Важно: принудительно включаем градиенты (полезно при рестартах)
        for param in teacher.parameters():
            param.requires_grad = True

    total_loss_accum, total_ls_accum, total_lt_accum, total_llat_accum = 0.0, 0.0, 0.0, 0.0
    grad_acc_steps = cfg.get("gradient_accumulation_steps", 2)
    optimizer.zero_grad()

    # 3. Расчет динамической Лямбды
    progress = epoch / cfg["epochs"]

    if progress <= peak_fraction:
        # Фаза 1: Плавный рост до max_lambda
        current_lambda = 1.0 + (cfg["max_lambda"] - 1.0) * (progress / peak_fraction)
    elif progress <= freeze_fraction:
        # Фаза 2: Плавное ослабление Учителя до min_lambda
        #decay_prog = (progress - peak_fraction) / (freeze_fraction - peak_fraction)
        #current_lambda = cfg["max_lambda"] - (cfg["max_lambda"] - min_lambda) * decay_prog
        current_lambda = cfg["max_lambda"]
    else:
        # Фаза 3: Фиксируем на минимальном уровне
        decay_prog = (progress - freeze_fraction) / (1.0 - freeze_fraction)
        current_lambda = cfg["max_lambda"] - (cfg["max_lambda"] - min_lambda) * decay_prog

    # 4. Основной цикл по батчам
    # ИСПРАВЛЕНИЕ 3: Достаем 3 элемента из лоадера
    for i, (xb, xf_b, yb_all) in enumerate(tqdm(loader, desc="train", leave=False)):
        xb = xb.to(device)
        xf_b = xf_b.to(device) # Будущее для Учителя
        y_meta = yb_all.to(device)
        target_rul = yb_all[:, 0].to(device) # RUL по-прежнему нужен Студенту как таргет

        with torch.amp.autocast('cuda'):
            # Студент не видит будущее
            s_preds, s_enc = student(xb, return_latents=True)
            # Учитель видит будущее через xf_b
            t_preds, t_enc = teacher(xb, x_future=xf_b, y_meta=y_meta, return_latents=True)

            loss_s = criterion(s_preds, target_rul)
            loss_t = criterion(t_preds, target_rul) if epoch <= warmup_epochs else torch.tensor(0.0, device=device)

            loss_lat = 0.0
            for se, te in zip(s_enc, t_enc):
                loss_lat += F.smooth_l1_loss(se, te.detach())
            loss_lat = loss_lat / len(s_enc)

            loss = (loss_s + loss_t + current_lambda * loss_lat) / grad_acc_steps

        scaler.scale(loss).backward()

        if (i + 1) % grad_acc_steps == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
            if epoch <= warmup_epochs:
                torch.nn.utils.clip_grad_norm_(teacher.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss_accum += loss.item() * grad_acc_steps
        total_ls_accum += loss_s.item()
        total_lt_accum += loss_t.item() if epoch <= warmup_epochs else 0.0
        total_llat_accum += loss_lat.item()

    num_batches = len(loader)

    for name, param in student.named_parameters():
        if 'projectors' in name and param.requires_grad:
            tb_writer.add_histogram(f'Weights/Student_Projector_{name}', param.clone().cpu().data.numpy(), epoch)

    return (total_loss_accum / num_batches, total_ls_accum / num_batches,
            total_lt_accum / num_batches, total_llat_accum / num_batches, current_lambda)


def evaluate(student, loader, device, max_rul=125):
    student.eval()
    ys, ps = [], []
    with torch.no_grad():
        # Добавляем распаковку xf_b (хоть мы его и не используем)
        for xb, xf_b, yb in loader: 
            xb, yb = xb.to(device), yb.to(device)
            preds = student(xb)

            if isinstance(preds, tuple):
                preds = preds[0]

            preds = preds.clamp(0.0, float(max_rul))
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())

    y_true, y_pred = np.concatenate(ys), np.concatenate(ps)

    res_rmse = rmse(y_true, y_pred)
    res_score = nasa_score(y_true, y_pred)

    print(f"Test RMSE: {res_rmse:.4f} | Max RUL: {y_pred.max():.2f}")

    return res_rmse, res_score, y_true, y_pred

# ---------------- Config ----------------
def get_fd_config(fd):

    data_base_dir = "/kaggle/input/datasets/mukhametovaskar/cmapss"
    save_base_dir = "/kaggle/working/"

    base = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "data_dir": data_base_dir,
        "save_dir": save_base_dir,
        "max_rul": 125,
        "seed": 44,
        "gradient_accumulation_steps": 2
    }

    fd_params = {
        1: {"window": 32, "batch_size": 32, "d_model": 128, "nhead": 1, "num_scales": 3, "lr": 0.00025, "epochs": 40, "patience": 40, "max_lambda": 300, 'future_len': 40},
        2: {"window": 64, "batch_size": 64, "d_model": 256,  "nhead": 4, "num_scales": 4, "lr": 0.0002, "epochs": 40, "patience": 40, "max_lambda": 200, 'future_len': 40},
        3: {"window": 48, "batch_size": 32, "d_model": 128, "nhead": 1, "num_scales": 3, "lr": 0.0002, "epochs": 40, "patience": 40, "max_lambda": 150, 'future_len': 40},
        4: {"window": 64, "batch_size": 64, "d_model": 256, "nhead": 4, "num_scales": 4, "lr": 0.0002, "epochs": 40, "patience": 40, "max_lambda": 300, 'future_len': 40},
    }

    if fd in [1]:
        base.update({"dropout": 0.05, "weight_decay": 1e-5, "ffn_dim": 512, "target_noise_std": 0.01, "lambda_warmup": 0.7})
    elif fd in [2]:
        base.update({"dropout": 0.14, "weight_decay": 5e-4, "ffn_dim": 1024, "target_noise_std": 0.03, "lambda_warmup": 0.7})
    elif fd in [3]:
        base.update({"dropout": 0.08, "weight_decay": 1e-5, "ffn_dim": 512, "target_noise_std": 0.02, "lambda_warmup": 0.7})
    elif fd in [4]:
        base.update({"dropout": 0.16, "weight_decay": 1e-4, "ffn_dim": 512, "target_noise_std": 0.03, "lambda_warmup": 0.7})

    base.update(fd_params[fd])
    base.setdefault("patch_size", 4)
    base.setdefault("pos_learnable", True)
    base.setdefault("optim_betas", (0.9, 0.999))
    base.setdefault("optim_eps", 1e-8)
    base.setdefault("encoder_layers_per_scale", 3)
    base.setdefault("decoder_layers_per_scale", 2)
    return base

In [4]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/

<IPython.core.display.Javascript object>

In [5]:
# ================= ГЛОБАЛЬНЫЕ НАСТРОЙКИ РЕЖИМА =================
FIND_BEST_EPOCH = True  # True: ищем эпоху на 10% валидации. False: финальный прогон на 100% трейна.
OPTIMAL_EPOCHS = {1: 21, 2: 32, 3: 12, 4: 6}  # Сюда впишешь найденные эпохи после режима True
# ===============================================================

In [6]:
# ---------------- Итоговый Main (Search + Final + Full Logging) ----------------
def main():
    from IPython.display import FileLink, display
    import zipfile

    torch.backends.cudnn.benchmark = True
    
    for fd in range(1, 5):
        cfg = get_fd_config(fd)
        set_seed(cfg["seed"])
        device = cfg["device"]

        # Оригинальные принты доступности GPU
        print(f"CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            try:
                print(f"GPU device name: {torch.cuda.get_device_name(0)}")
            except: pass

        print(f"\n=== TRAIN FD00{fd} cfg: {cfg} ===")
        os.makedirs(cfg["save_dir"], exist_ok=True)
        scaler = torch.amp.GradScaler('cuda', enabled=(device != 'cpu'))

        train_path = f"{cfg['data_dir']}/train_FD00{fd}.txt"
        test_path  = f"{cfg['data_dir']}/test_FD00{fd}.txt"
        rul_path   = f"{cfg['data_dir']}/RUL_FD00{fd}.txt"

        if not os.path.exists(train_path):
            raise FileNotFoundError(f"Файл датасета не найден по пути: {train_path}. Проверь пути в get_fd_config!")

        df_train = load_cmapss(train_path)
        df_test = load_cmapss(test_path)

        # Нормализация
        if fd in [2, 4]:
            print(f"Applying Regime-Specific Normalization for FD00{fd} (6 clusters)...")
            df_train, df_test = apply_regime_specific_normalization(df_train, df_test, n_clusters=6)
            n_clusters=6
        else:
            print(f"Applying Global Normalization for FD00{fd} (or single cluster)...")
            df_train, df_test = apply_regime_specific_normalization(df_train, df_test, n_clusters=1)
            n_clusters=1

        # ================= РАЗБИЕНИЕ НА ВАЛИДАЦИЮ / ФИНАЛЬНЫЙ ПРОГОН =================
        engine_ids_train = df_train['id'].unique()
        num_val_engines = max(1, int(len(engine_ids_train) * 0.1)) 

        if FIND_BEST_EPOCH:
            print(f"\n--- 🔍 РЕЖИМ ПОИСКА ЭПОХИ (Validation Mode) ---")
            val_ids = engine_ids_train[-num_val_engines:]
            train_ids = engine_ids_train[:-num_val_engines]
            
            df_val_full = df_train[df_train['id'].isin(val_ids)].copy()
            df_train_actual = df_train[df_train['id'].isin(train_ids)].copy()
            
            val_ruls = []
            df_val_cut_list = []
            for eid in val_ids:
                sub = df_val_full[df_val_full['id'] == eid].sort_values('cycle')
                T = len(sub)
                cut_point = random.randint(cfg['window'], T - 1) if T > cfg['window'] else T
                df_val_cut_list.append(sub.iloc[:cut_point])
                val_ruls.append(T - cut_point)
                
            df_val = pd.concat(df_val_cut_list)
            df_val_rul = pd.DataFrame(val_ruls)
            eval_prefix = "Val" # Префикс для логов
            print(f"Трейн: {len(train_ids)} моторов. Валидация: {len(val_ids)} моторов.")
        else:
            print(f"\n--- 🚀 ФИНАЛЬНЫЙ ПРОГОН (Full Train Mode) ---")
            print(f"Обучаемся на 100% данных ровно {OPTIMAL_EPOCHS[fd]} эпох.")
            cfg['epochs'] = OPTIMAL_EPOCHS[fd]
            df_train_actual = df_train
            eval_prefix = "Test"

        future_len = cfg['future_len']

        # Подготовка данных (Train)
        X_tr, X_fut_tr, y_tr_meta = create_train_windows_v4(df_train_actual, cfg["window"], future_len=future_len, max_rul=cfg["max_rul"], stride=1, fd_num=fd)
        print(f"Train windows: {X_tr.shape} | Future: {X_fut_tr.shape} | Meta: {y_tr_meta.shape}")

        # Подготовка данных (Test/Val)
        df_rul_test = pd.read_csv(rul_path, sep=r'\s+', header=None)
        X_test, y_test = create_test_windows(df_test, df_rul_test, cfg["window"], cfg["max_rul"])
        X_fut_test = np.zeros((X_test.shape[0], future_len, X_tr.shape[-1]), dtype=np.float32)

        train_ds = CMapssWindowDataset(X_tr, X_fut_tr, y_tr_meta)
        test_ds  = CMapssWindowDataset(X_test, X_fut_test, y_test)

        pin_mem = torch.cuda.is_available()
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, pin_memory=pin_mem, num_workers=2)
        test_loader  = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, pin_memory=pin_mem, num_workers=2)

        # Лоадер для оценки (выбираем Val или Test)
        if FIND_BEST_EPOCH:
            X_val, y_val = create_test_windows(df_val, df_val_rul, cfg["window"], cfg["max_rul"])
            X_fut_val = np.zeros((X_val.shape[0], future_len, X_tr.shape[-1]), dtype=np.float32)
            eval_ds = CMapssWindowDataset(X_val, X_fut_val, y_val)
            eval_loader = DataLoader(eval_ds, batch_size=cfg["batch_size"], shuffle=False, pin_memory=pin_mem, num_workers=2)
        else:
            eval_loader = test_loader

        # 3. Модель
        n_sensors = X_tr.shape[-1]
        model_kwargs = {
            "window": cfg["window"], "n_sensors": n_sensors, "d_model": cfg["d_model"],
            "nhead": cfg["nhead"], "num_scales": cfg["num_scales"], "ffn_dim": cfg["ffn_dim"],
            "patch_size": cfg["patch_size"], "dropout": cfg["dropout"],
            "encoder_layers_per_scale": cfg["encoder_layers_per_scale"],
            "decoder_layers_per_scale": cfg["decoder_layers_per_scale"],
            "pos_learnable": cfg["pos_learnable"], "target_noise_std": cfg["target_noise_std"],
            "num_regimes": n_clusters, "num_faults": 2 if fd in [3, 4] else 1, "future_len": future_len
        }

        student = STARModelFull(**model_kwargs, is_teacher=False).to(device)
        teacher = STARModelFull(**model_kwargs, is_teacher=True).to(device)

        # 4. Оптимизация
        optimizer = optim.Adam(list(student.parameters()) + list(teacher.parameters()),
                               lr=cfg["lr"], betas=cfg["optim_betas"], eps=cfg["optim_eps"], weight_decay=cfg["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=1e-5)
        criterion = nn.MSELoss()

        best_eval_rmse = 1e9

        # Параметры для визуализации траекторий (всегда по df_test для контроля)
        engine_ids_test = [int(e) for e in sorted(df_test['id'].unique())]
        engine_rul_test = dict(zip(engine_ids_test, df_rul_test.values.flatten()))
        selected_engines = {
            "low": min(engine_rul_test, key=engine_rul_test.get),
            "high": max(engine_rul_test, key=engine_rul_test.get),
            "random": random.choice(engine_ids_test)
        }
        print("Selected engines for visualization:", selected_engines)

        # --- ИНИЦИАЛИЗАЦИЯ ЛОГГЕРОВ ---
        log_dir = os.path.join(cfg["save_dir"], f"tb_fd00{fd}_{eval_prefix}")
        tb_writer = SummaryWriter(log_dir)
        csv_path = os.path.join(cfg["save_dir"], f"metrics_fd00{fd}_{eval_prefix}.csv")

        with open(csv_path, mode='w', newline='') as csv_file:
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow(['Epoch', 'Train_Loss', 'Loss_Student', 'Loss_Teacher', 'Loss_Latent', 'Lambda', f'{eval_prefix}_RMSE', f'{eval_prefix}_Score'])

            for epoch in range(1, cfg["epochs"] + 1):
                print(f"\nEpoch {epoch}/{cfg['epochs']}")

                # Обучение
                t_loss, l_stud, l_teach, l_lat, cur_lam = train_one_epoch(
                    student, teacher, train_loader, optimizer, device, scaler, criterion, cfg, epoch, tb_writer, fd
                )

                # Оценка (на eval_loader)
                cur_rmse, cur_score, y_true_all, y_pred_all = evaluate(student, eval_loader, device, cfg["max_rul"])
                
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']
                
                # ОРИГИНАЛЬНЫЕ ПРИНТЫ
                print(f"Epoch {epoch}: Learning Rate = {current_lr}")
                print(f"Loss: {t_loss:.2f} | L_s(Stud): {l_stud:.2f} | L_t(Teach): {l_teach:.2f} | L_lat: {l_lat:.2f} | (λ={cur_lam:.3f})")
                print(f"{eval_prefix} RMSE: {cur_rmse:.4f} | Score: {cur_score:.4f}")

                # --- ЗАПИСЬ В TENSORBOARD ---
                tb_writer.add_scalar('Loss/Total', t_loss, epoch)
                tb_writer.add_scalar('Loss/Student_MSE', l_stud, epoch)
                tb_writer.add_scalar('Loss/Teacher_MSE', l_teach, epoch)
                tb_writer.add_scalar('Loss/Latent_SmoothL1', l_lat, epoch)
                tb_writer.add_scalar(f'Metrics/{eval_prefix}_RMSE', cur_rmse, epoch)
                tb_writer.add_scalar(f'Metrics/{eval_prefix}_NASA_Score', cur_score, epoch)
                tb_writer.add_scalar('Hyperparams/Lambda', cur_lam, epoch)

                # --- ВИЗУАЛИЗАЦИЯ SCATTER ---
                fig_scatter = plot_scatter_rul(y_true_all, y_pred_all, epoch, fd)
                tb_writer.add_figure(f'Analysis/{eval_prefix}_Scatter', fig_scatter, epoch)
                plt.close(fig_scatter)

                # --- ВИЗУАЛИЗАЦИЯ ТРАЕКТОРИЙ ---
                for tag, eid in selected_engines.items():
                    y_true_traj, y_pred_traj = get_engine_trajectory(
                        student, df_test, engine_rul_test, engine_idx=eid, window=cfg["window"],
                        max_rul=cfg["max_rul"], device=device
                    )
                    fig_engine = plot_engine_prediction(y_true_traj, y_pred_traj, engine_idx=eid, epoch=epoch, fd=fd)
                    tb_writer.add_figure(f'Engine/{tag}_engine_{eid}', fig_engine, epoch)
                    plt.close(fig_engine)

                # --- ВИЗУАЛИЗАЦИЯ LATENT SPACE (t-SNE) ---
                if epoch % 5 == 0 or epoch == cfg["epochs"]:
                    student.eval(); teacher.eval()
                    xb_train, xf_train, yb_meta_train = next(iter(train_loader))
                    with torch.no_grad():
                        _, s_lat = student(xb_train.to(device), return_latents=True)
                        _, t_lat = teacher(xb_train.to(device), x_future=xf_train.to(device), y_meta=yb_meta_train.to(device), return_latents=True)
                        s_vec = s_lat[-1].mean(dim=(1,2)).cpu().numpy()
                        t_vec = t_lat[-1].mean(dim=(1,2)).cpu().numpy()
                    fig_tsne = plot_latent_space_tsne(s_vec, t_vec, epoch, fd)
                    tb_writer.add_figure('Latent/tSNE', fig_tsne, epoch)
                    plt.close(fig_tsne)

                # --- ЗАПИСЬ В CSV ---
                csv_writer.writerow([epoch, t_loss, l_stud, l_teach, l_lat, cur_lam, cur_rmse, cur_score])
                csv_file.flush()
                tb_writer.flush()

                # Сохранение лучшей модели
                if cur_rmse < best_eval_rmse:
                    best_eval_rmse = cur_rmse
                    print(f"★ New best {eval_prefix} RMSE {best_eval_rmse:.4f} found. Saving model.")
                    torch.save({"model": student.state_dict(), "cfg": cfg, "rmse": best_eval_rmse},
                               os.path.join(cfg["save_dir"], f"best_star_fd{fd}.pth"))

            # Сохранение последней эпохи
            torch.save({"model": student.state_dict(), "cfg": cfg},
                       os.path.join(cfg["save_dir"], f"last_star_fd{fd}.pth"))

        tb_writer.close()
        print(f"=== FD00{fd} finished. Best {eval_prefix} RMSE: {best_eval_rmse:.4f} ===\n")

    # --- Создание ZIP архива ---
    save_dir = cfg["save_dir"]
    zip_name = 'all_experiment_results.zip'
    print(f"\nУпаковка результатов из {save_dir} в архив...")
    
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(save_dir):
            for file in files:
                if file.endswith(('.pth', '.csv')) or 'events' in file:
                    file_path = os.path.join(root, file)
                    arc_name = os.path.relpath(file_path, os.path.dirname(save_dir))
                    zipf.write(file_path, arcname=arc_name)

    display(FileLink(zip_name))
    print(f"\nАрхив {zip_name} создан!")

In [7]:
if __name__ == "__main__":
    main()

CUDA available: True
GPU device name: Tesla T4

=== TRAIN FD001 cfg: {'device': 'cuda', 'data_dir': '/kaggle/input/datasets/mukhametovaskar/cmapss', 'save_dir': '/kaggle/working/', 'max_rul': 125, 'seed': 44, 'gradient_accumulation_steps': 2, 'dropout': 0.05, 'weight_decay': 1e-05, 'ffn_dim': 512, 'target_noise_std': 0.01, 'lambda_warmup': 0.7, 'window': 32, 'batch_size': 32, 'd_model': 128, 'nhead': 1, 'num_scales': 3, 'lr': 0.00025, 'epochs': 40, 'patience': 40, 'max_lambda': 300, 'future_len': 40, 'patch_size': 4, 'pos_learnable': True, 'optim_betas': (0.9, 0.999), 'optim_eps': 1e-08, 'encoder_layers_per_scale': 3, 'decoder_layers_per_scale': 2} ===
Applying Global Normalization for FD001 (or single cluster)...

--- 🔍 РЕЖИМ ПОИСКА ЭПОХИ (Validation Mode) ---
Трейн: 90 моторов. Валидация: 10 моторов.
Train windows: (15590, 32, 14) | Future: (15590, 40, 14) | Meta: (15590, 3)
Selected engines for visualization: {'low': 34, 'high': 25, 'random': 4}

Epoch 1/40


Test RMSE: 23.7825 | Max RUL: 114.05
Epoch 1: Learning Rate = 0.0002496300800479754
Loss: 3453.01 | L_s(Stud): 1977.20 | L_t(Teach): 1471.11 | L_lat: 0.29 | (λ=15.950)
Val RMSE: 23.7825 | Score: 149.3323
★ New best Val RMSE 23.7825 found. Saving model.

Epoch 2/40


Test RMSE: 21.9670 | Max RUL: 120.79
Epoch 2: Learning Rate = 0.0002485226008714166
Loss: 500.92 | L_s(Stud): 419.37 | L_t(Teach): 77.53 | L_lat: 0.13 | (λ=30.900)
Val RMSE: 21.9670 | Score: 164.6454
★ New best Val RMSE 21.9670 found. Saving model.

Epoch 3/40


Test RMSE: 15.9093 | Max RUL: 120.14
Epoch 3: Learning Rate = 0.00024668439044772124
Loss: 397.43 | L_s(Stud): 325.84 | L_t(Teach): 66.72 | L_lat: 0.11 | (λ=45.850)
Val RMSE: 15.9093 | Score: 33.8979
★ New best Val RMSE 15.9093 found. Saving model.

Epoch 4/40


Test RMSE: 18.4978 | Max RUL: 124.05
Epoch 4: Learning Rate = 0.0002441267819554185
Loss: 335.08 | L_s(Stud): 270.21 | L_t(Teach): 60.09 | L_lat: 0.08 | (λ=60.800)
Val RMSE: 18.4978 | Score: 74.3739

Epoch 5/40


...

Test RMSE: 15.0076 | Max RUL: 124.19
Epoch 38: Learning Rate = 1.1169607643461924e-05
Loss: 126.56 | L_s(Stud): 123.50 | L_t(Teach): 0.00 | L_lat: 0.02 | (λ=140.000)
Val RMSE: 15.0076 | Score: 126.9185

Epoch 39/40


Test RMSE: 15.1511 | Max RUL: 124.30
Epoch 39: Learning Rate = 1.0292853295352844e-05
Loss: 124.55 | L_s(Stud): 121.92 | L_t(Teach): 0.00 | L_lat: 0.02 | (λ=120.000)
Val RMSE: 15.1511 | Score: 141.2141

Epoch 40/40


Test RMSE: 14.6638 | Max RUL: 123.93
Epoch 40: Learning Rate = 1e-05
Loss: 124.75 | L_s(Stud): 122.54 | L_t(Teach): 0.00 | L_lat: 0.02 | (λ=100.000)
Val RMSE: 14.6638 | Score: 119.6591
=== FD004 finished. Best Val RMSE: 11.1093 ===


Упаковка результатов из /kaggle/working/ в архив...


/kaggle/working/all_experiment_results.zip


Архив all_experiment_results.zip создан!
